In [ ]:
# Базовые библиотеки для воспроизводимости, анализа и удобного вывода результатов.
import os
import re
import sys
import random
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def safe_ensure_package(package_name: str, import_name: Optional[str] = None) -> bool:
    """Пытается импортировать пакет и при необходимости установить его через pip.
    Если установка не удалась, возвращает False, но не роняет ноутбук.
    """
    target = import_name or package_name
    try:
        __import__(target)
        return True
    except Exception:
        print(f"Пробуем установить пакет: {package_name}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
            __import__(target)
            return True
        except Exception as e:
            print(f"Не удалось подготовить пакет {package_name}: {e!r}")
            return False


FAISS_READY = safe_ensure_package("faiss-cpu", "faiss")

try:
    import faiss  # type: ignore
except Exception:
    faiss = None
    FAISS_READY = False


# sentence-transformers опционален: ноутбук умеет работать и без него.
SENTENCE_TRANSFORMERS_READY = safe_ensure_package("sentence-transformers", "sentence_transformers")

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS доступен:", FAISS_READY)
print("sentence-transformers доступен:", SENTENCE_TRANSFORMERS_READY)

In [ ]:
# Фиксируем seed и определяем устройство.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)


set_seed(42)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("Устройство для работы:", DEVICE)

In [ ]:
# Учебная база знаний.
documents: List[Dict[str, str]] = [
    {
        "doc_id": "doc_01",
        "title": "Эмбеддинги текстов",
        "text": (
            "Эмбеддинг – это векторное представление текста. "
            "Семантически похожие фразы оказываются близко в векторном пространстве. "
            "Это позволяет искать документы не только по совпадению слов, но и по смыслу запроса."
        ),
    },
    {
        "doc_id": "doc_02",
        "title": "FAISS и поиск ближайших соседей",
        "text": (
            "FAISS – библиотека для быстрого поиска ближайших соседей по векторам. "
            "Она полезна, когда в базе знаний много чанков и нужно быстро находить top-k наиболее похожих фрагментов."
        ),
    },
    {
        "doc_id": "doc_03",
        "title": "Чанкинг и overlap",
        "text": (
            "Чанкинг разбивает длинный документ на более короткие фрагменты. "
            "Если чанк слишком большой, retrieval может возвращать слишком общий контекст. "
            "Если чанк слишком маленький, смысл распадается. "
            "Overlap помогает не потерять информацию на границах соседних фрагментов."
        ),
    },
    {
        "doc_id": "doc_04",
        "title": "Оценка качества retrieval",
        "text": (
            "Качество retrieval нельзя оценивать только визуально. "
            "Обычно используют hit@k, recall@k и MRR. "
            "Эти метрики помогают понять, нашел ли retriever релевантный документ и насколько высоко он оказался в выдаче."
        ),
    },
    {
        "doc_id": "doc_05",
        "title": "Обновление базы знаний",
        "text": (
            "После добавления новых документов база знаний должна быть переиндексирована. "
            "Иначе retriever не увидит новые фрагменты. "
            "В production-системах обновление индекса может быть периодическим или событийным."
        ),
    },
    {
        "doc_id": "doc_06",
        "title": "Галлюцинации в RAG",
        "text": (
            "RAG снижает риск галлюцинаций, но не устраняет его полностью. "
            "Если retrieval вернул нерелевантные фрагменты или генератор исказил найденный факт, итоговый ответ все равно будет ошибочным."
        ),
    },
    {
        "doc_id": "doc_07",
        "title": "Промпт с контекстом",
        "text": (
            "В RAG-сценарии важно явно передавать модели найденный контекст и ограничивать ответ этим контекстом. "
            "Полезно просить систему отвечать только на основании источников и возвращать указание на использованные фрагменты."
        ),
    },
    {
        "doc_id": "doc_08",
        "title": "Метаданные и фильтрация",
        "text": (
            "Помимо текста, retrieval может учитывать метаданные: тип документа, дату, автора, подразделение или теги. "
            "Фильтрация по метаданным уменьшает область поиска и повышает точность извлечения."
        ),
    },
    {
        "doc_id": "doc_09",
        "title": "Реранжирование результатов",
        "text": (
            "После первичного retrieval можно применять реранжирование. "
            "Сначала быстрый retriever извлекает кандидатов, а затем более точная модель пересортировывает их по полезности для ответа."
        ),
    },
    {
        "doc_id": "doc_10",
        "title": "Гибридный поиск",
        "text": (
            "Гибридный поиск сочетает dense retrieval и классический лексический поиск. "
            "Он полезен, когда часть запросов требует смыслового сходства, а часть – точного совпадения терминов или аббревиатур."
        ),
    },
]

docs_df = pd.DataFrame(documents)
display(docs_df[["doc_id", "title"]])

In [ ]:
def chunk_text(text: str, chunk_size: int = 28, overlap: int = 8) -> List[str]:
    words = text.split()
    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]
        if not chunk_words:
            continue
        chunks.append(" ".join(chunk_words))
        if end >= len(words):
            break
    return chunks


class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError


class TfidfBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.backend_name = "TF-IDF (fallback)"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        matrix = self.vectorizer.fit_transform(texts)
        vectors = matrix.astype(np.float32).toarray()
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        matrix = self.vectorizer.transform(texts)
        vectors = matrix.astype(np.float32).toarray()
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms


class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore
        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        return vectors.astype(np.float32)

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        return vectors.astype(np.float32)


def choose_backend(device: str = "cpu") -> EmbeddingBackend:
    # Опциональная попытка dense backend.
    if SENTENCE_TRANSFORMERS_READY:
        try:
            return SentenceTransformersBackend(
                model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
                device=device,
            )
        except Exception as e:
            print("Dense backend недоступен, переходим к TF-IDF.")
            print("Причина:", repr(e))
    return TfidfBackend()


@dataclass
class RetrieverArtifacts:
    backend_name: str
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    backend: EmbeddingBackend
    index: object


def build_retriever(
    documents: List[Dict[str, str]],
    chunk_size: int = 28,
    overlap: int = 8,
    device: str = "cpu",
) -> RetrieverArtifacts:
    rows = []
    for doc in documents:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_id, chunk_text_value in enumerate(chunks, start=1):
            rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "chunk_id": f'{doc["doc_id"]}_chunk_{chunk_id:02d}',
                    "chunk_text": chunk_text_value,
                }
            )

    chunks_df = pd.DataFrame(rows)
    backend = choose_backend(device=device)
    chunk_vectors = backend.fit_documents(chunks_df["chunk_text"].tolist()).astype(np.float32)

    if FAISS_READY:
        index = faiss.IndexFlatIP(chunk_vectors.shape[1])  # type: ignore
        index.add(chunk_vectors)
    else:
        index = chunk_vectors

    return RetrieverArtifacts(
        backend_name=backend.backend_name,
        chunks_df=chunks_df,
        chunk_vectors=chunk_vectors,
        backend=backend,
        index=index,
    )


def search_chunks(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> pd.DataFrame:
    query_vector = artifacts.backend.encode_queries([query]).astype(np.float32)

    if FAISS_READY:
        scores, indices = artifacts.index.search(query_vector, top_k)  # type: ignore
        scores = scores[0]
        indices = indices[0]
    else:
        similarities = (artifacts.chunk_vectors @ query_vector.T).reshape(-1)
        indices = np.argsort(-similarities)[:top_k]
        scores = similarities[indices]

    result = artifacts.chunks_df.iloc[indices].copy().reset_index(drop=True)
    result.insert(0, "rank", np.arange(1, len(result) + 1))
    result["score"] = scores
    return result[["rank", "score", "doc_id", "title", "chunk_id", "chunk_text"]]